In [9]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from IPython.display import Markdown
from dotenv.ipython import load_dotenv

In [10]:
load_dotenv(override=True)

True

In [11]:
llm=ChatOpenAI(model="gpt-4o",temperature=0)

In [12]:
agent = create_agent(
model=llm,
system_prompt= "You are a helpful assistant"
)

In [13]:
agent = create_agent(
model="openai:gpt-4o",
tools=[],
system_prompt= "You are a helpful assistant"
)

In [14]:
respt = agent.invoke(input={"messages":
[{'role':'user', 'content':'je m appelle douaa'}]})

In [15]:
print(display(Markdown(respt['messages'][-1].content)))

Bonjour Douaa ! Comment puis-je vous aider aujourd'hui ?

None


In [16]:
from langchain.agents.middleware import wrap_model_call,ModelRequest, ModelResponse
from langchain.messages import HumanMessage,SystemMessage, AIMessage


In [17]:
basic_llm=ChatOpenAI(model="gpt-4o-mini",temperature=0)
advanced_llm=ChatOpenAI(model="gpt-4o",temperature=0)

In [18]:
@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    env = request.state.get("env", "test")
    print("ENV:", env)

    if env == "prod":
        print("advanced model selected")
        request.model = advanced_llm
    else:
        print("basic model selected")
        request.model = basic_llm

    return handler(request)

In [19]:
agent2 = create_agent(
model=basic_llm, 
tools=[],
middleware=[dynamic_model_selection],
debug=True
)

In [20]:
resp = agent2.invoke(
    {
        "messages": [HumanMessage(content="c'est quoi un agent ai?")],
        "env": "prod"  
    }
)
print(resp["messages"][-1].content)

[values] {'messages': [HumanMessage(content="c'est quoi un agent ai?", additional_kwargs={}, response_metadata={}, id='7d23b19e-7781-4b53-919c-fe6cd554c63a')]}
ENV: test
basic model selected


C:\Users\douaa\AppData\Local\Temp\ipykernel_30164\3921408860.py:11: DeprecationWarning: Direct attribute assignment to ModelRequest.model is deprecated. Use request.override(model=...) instead to create a new request with the modified attribute.
  request.model = basic_llm


[updates] {'model': {'messages': [AIMessage(content="Un agent AI (intelligence artificielle) est un système ou un programme informatique capable d'effectuer des tâches qui nécessitent normalement une intelligence humaine. Ces agents peuvent percevoir leur environnement, prendre des décisions, apprendre de nouvelles informations et interagir avec les utilisateurs ou d'autres systèmes.\n\nIl existe différents types d'agents AI, notamment :\n\n1. **Agents réactifs** : Ils réagissent à des stimuli dans leur environnement sans mémoire ou capacité d'apprentissage. Par exemple, un programme qui joue aux échecs en suivant des règles prédéfinies.\n\n2. **Agents basés sur des modèles** : Ils utilisent des modèles internes pour représenter leur environnement et prendre des décisions basées sur ces modèles.\n\n3. **Agents d'apprentissage** : Ces agents peuvent apprendre de leurs expériences et améliorer leurs performances au fil du temps, comme les systèmes de recommandation ou les assistants virt

In [21]:
from langchain.agents import AgentState
from langchain.agents.middleware import AgentMiddleware
from typing import Any
from langgraph.checkpoint.memory import InMemorySaver

In [22]:
agent = create_agent(
model="openai:gpt-4o",
system_prompt="You are a helpful assistant",
checkpointer=InMemorySaver()
)

In [23]:
res=agent.invoke(
input={'messages':[HumanMessage('My Name is Mohamed')]},
config={'configurable':{'thread_id':1}}
)
print(res['messages'][-1].content)

Hello, Mohamed! How can I assist you today?


In [24]:
res=agent.invoke(
input={'messages':[HumanMessage('What is my name')]},
config={'configurable':{'thread_id':1}}
)
print(res['messages'][-1].content)

Your name is Mohamed. How can I help you today?


In [25]:
from langchain.tools import tool
from langchain.agents import create_agent

In [26]:
@tool
def get_weather(city: str) -> str:
  """Get weather information for a city."""
  print(f'Weather tool is called for {city}')
  return {
    'city':city,
    'temperature': 32,
    'humidity': 60,
    'condition': 'Sunny'
  }
@tool
def get_employee_info(name: str) -> str:
  """Get information aboud the given employee name"""
  print(f'get_employee_info tool is called for {name}')
  return {'name' : name, 
          'salary': 12000, 
          'job': 'Developper'}

In [27]:
agent4 = create_agent(
    model="openai:gpt-4o",
    tools=[get_weather, get_employee_info],
    checkpointer=InMemorySaver(),
    system_prompt="You are a helpful assistant",
)

In [32]:
resp = agent4.invoke(
    {"messages": [HumanMessage("What is the weather in Marrakech?")]},
    config={
        "configurable": {
            "thread_id": "1" 
        }
    }
)
print(resp["messages"][-1].content)

The weather in Marrakech is currently sunny, with a temperature of 32°C and a humidity level of 60%.


In [36]:
load_dotenv(override=True)

True

In [37]:
from langchain_tavily import TavilySearch

In [38]:
tavily = TavilySearch(max_results=10, search_depth="advanced")

In [39]:
@tool
def searh_web(query: str):
    """
    Search the web for the given query.

    """
    print(f"Searching the web for: {query}")
    result = tavily.invoke({"query": query})
    return result

In [40]:
agent5 = create_agent(
    model="openai:gpt-4o",
    tools=[get_weather, get_employee_info],
    checkpointer=InMemorySaver(),
    system_prompt="You are a helpful assistant"

)

In [41]:
response = agent5.invoke(
    input={'messages':[HumanMessage('What is the weather in Marrakech')]},
    config={'configurable': {'thread_id': '1'}}
)
print(response)

Weather tool is called for Marrakech
{'messages': [HumanMessage(content='What is the weather in Marrakech', additional_kwargs={}, response_metadata={}, id='6c59076a-c422-4796-964f-9c5dec99c6ce'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 80, 'total_tokens': 96, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_d4e5e0e07a', 'id': 'chatcmpl-DSfIkmDYaKpRiIz2Y4zfmG2bmUeMh', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d716a-cc3a-7241-95d6-d4ef5b9b4ad7-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Marrakech'}, 'id': 'call_oRYawuSI8ys2lAzbSVzoq03U', 'type': 'tool_call'}], invalid_tool_calls=[], usage_

In [42]:
from langchain_experimental.tools import PythonREPLTool

In [43]:
repl_tool = PythonREPLTool()

In [44]:
agent6 = create_agent(
    model="openai:gpt-4o",
    tools=[repl_tool],
    system_prompt="genere et execute du code python en placant le code python et le resultat dans le fichier doc.txt"

)

In [45]:
res = agent6.invoke(input={"messages":[
    HumanMessage("je veux trier les deux liste [6,8,9,0] et [6,3,7,0] et puis aditionner les deux listes")
]}) 


Python REPL can execute arbitrary code. Use with caution.
Python REPL can execute arbitrary code. Use with caution.
